# 1. Build a guideline chatbot (RAG)

## Why an LLM session in an MLOps workshop?

Everything so far (malnutrition risk, TB screening) had the same shape: a model outputs a **number or a class**,
and later we learn the **true label** (did the child become malnourished? did the lab confirm TB?). So we could
count hits and misses: sensitivity, ROC-AUC, calibration.

An LLM application breaks that shape:

| | Classic model (EMR, TB) | LLM application (this session) |
|---|---|---|
| Output | a risk score or a class | free text: an answer in sentences |
| "Right answer" | one true label, known later | many acceptable answers; partly right is common |
| How we score it | compare with the label: sensitivity, ROC-AUC | **no label to compare with**: someone (or a model) has to *judge* the text |
| Same input twice | same output | can differ (the model samples words) |
| What is versioned | data, code, model | documents, parser, chunking, search, **prompt**, model, and a hosted model can change under you |
| New ways to fail | wrong class | invents facts, misreads a table, follows an instruction planted in a document |
| Cost | ~free per prediction | paid per call, seconds per answer |

For **clinical decision support** the stakes are higher still: a fluent, confident answer that the guideline
does not support is more dangerous than a model that says "I don't know".

So we need new ways to evaluate, and new things to version and monitor. That is today.

## The plan

1. **This notebook:** build a chatbot that answers from six Ethiopian public health guidelines, step by step, and
   play with it in a chat page.
2. **Notebook 2:** how do you evaluate it? Run 24 questions through it and let an LLM judge (Ragas) score it.
3. **Notebook 3:** change one part at a time (parsing, chunking, search), re-run the same evaluation, compare.

The approach is called **retrieval-augmented generation (RAG)**: first *retrieve* the guideline passages that
match the question, then let the LLM *generate* an answer from those passages only.

In [1]:
import os
import re
import textwrap
from pathlib import Path

import pandas as pd

import rag                                   # the finished version of what we build here, for later

pd.set_option("display.width", 200, "display.max_colwidth", 120)
assert os.environ.get("OPENROUTER_API_KEY"), "Set OPENROUTER_API_KEY (the shared workshop key) and restart the kernel"
client = rag.llm_client()                    # an OpenAI-compatible client pointed at OpenRouter
MODEL = "google/gemma-4-26b-a4b-it"          # an open-weight model (Apache-2.0), hosted on OpenRouter


def show(text, width=110):
    print(textwrap.fill(text, width) if "\n" not in text else text)

## Step 0. Why not just ask the model?

Ask the model a question about the cholera guideline, with no documents at all.

In [2]:
question = "According to Ethiopia's national cholera guideline, what is a cholera alert, and how quickly must it be reported?"
reply = client.chat.completions.create(model=MODEL, temperature=0, **rag.thinking_switch(False),
                                       messages=[{"role": "user", "content": question}])
show(reply.choices[0].message.content)

Based on Ethiopia's national guidelines for cholera management and surveillance, the definitions and reporting requirements are as follows:

### **What is a Cholera Alert?**
A **cholera alert** is triggered when there is a **single confirmed case** of *Vibrio cholerae* (the bacterium that causes cholera) or a **cluster of suspected cases** (patients presenting with acute watery diarrhea) in a specific area.

In the context of Ethiopian surveillance, an alert is often categorized by the level of suspicion:
1.  **Suspected Case:** Any patient presenting with acute watery diarrhea (AWD) that could be cholera.
2.  **Confirmed Case:** A case where the bacterium *Vibrio cholerae* has been laboratory-confirmed.

An "alert" status is typically declared to initiate immediate public health actions, such as investigation, contact tracing, and enhanced surveillance, even before laboratory confirmation is finalized, to prevent a widespread outbreak.

### **How quickly must it be reported?**
Ethiopi

> **PAUSE (2 min).** Is that right? How would you know? Where did the model get it from?

The model answers from whatever it absorbed in training: maybe right, maybe another country's rule, maybe an old
version, maybe invented. It cannot show you a page. The guideline says (page 14): *two or more people aged 2 years
or older with acute watery diarrhea and severe dehydration, or dying from it, in the same area within one week;
report immediately, within 30 minutes.*

RAG fixes the "where from?" problem: give the model the right pages and tell it to answer only from them.

```
           ┌── once ─────────────────────────────────────────┐
 PDFs ──►  │ 1. PARSE: PDF → text   2. CHUNK: text → passages │ ──► passages
           │ 3. INDEX: make the passages searchable          │
           └─────────────────────────────────────────────────┘
 question ──► 4. RETRIEVE: the k best passages ──► 5. GENERATE: LLM answers from them, citing [1], [2] ──► answer
```

We now build each step ourselves.

## Step 1. Parse: from PDF to text

A PDF is made for printing, not for reading by a program: it stores characters at positions on a page. Getting
text out of it is called **parsing**. The simplest parser, `pypdf`, reads the characters in order. Here is page 38
of the cholera guideline, which has a table of antibiotic doses.

In [3]:
from pypdf import PdfReader

PDF = rag.DOCS / "ephi_cholera_2022.pdf"         # downloaded during setup (scripts/download.py guidelines)
PAGE = 38
pdf_text = PdfReader(PDF).pages[PAGE - 1].extract_text()
print(pdf_text)

31
Guideline for Cholera Surveillance and 
Outbreak Response Ethiopia
4 hours of treatment) or treatment 
failure (the patient is still dehydrated 
after completing the initial 4 hours of 
rehydration therapy), regardless of the 
degree of dehydration; and
 ■ Patients with coexisting conditions 
(including pregnancy) or comorbidities 
(such as SAM, HIV), regardless of the 
degree of dehydration. 
Antibiotics are given as soon as the patient 
is able to take oral medication (once vomiting 
has stopped). 
 ■ Doxycycline — single dose (300 mg for 
adults; 2–4 mg/kg for children under 12 
years of age) is the antibiotic of choice 
for all patients, including pregnant 
women. 
 ■ If resistance to doxycycline is 
documented, give azithromycin 1 g or 
ciprofloxacin 1 g orally as a single dose 
for adults. 
 ■ For children under 12 years of age, give 
azithromycin 20 mg/kg (max 1 g) or 
ciprofloxacin 20 mg/kg (max 1 g) orally 
as a single dose
Age group First- line Alternative
Adults (includin

Look at the table at the bottom ("Age group | First-line | Alternative"): pypdf gives the cells in reading order,
so rows and columns are run together. A human can still guess; a search engine and an LLM may not.

A **layout-aware parser** looks at the page as an image as well as text and rebuilds the structure. We used
**LlamaParse** (a commercial service, which uses vision-language models) once, before the workshop, to turn all six
guidelines into **Markdown**: headings as `#`, tables as `| rows |`, running headers and page numbers removed, and
a marker `<!-- page 38 -->` at the start of every page so we can still cite pages. It costs money and needs a key,
so the result is saved in `guidelines_md/` (settings and quality checks: `data/SOURCES.md`).

The same page, as LlamaParse returned it:

In [4]:
def markdown_page(doc, page):
    """The Markdown LlamaParse produced for one PDF page (between its page markers)."""
    text = (rag.DOCS_MD / f"{doc}.md").read_text()
    pages = re.split(r"<!-- page (\d+)[^>]*-->", text)          # [before, n, text, n, text, ...]
    return dict(zip(map(int, pages[1::2]), pages[2::2]))[page].strip()


print(markdown_page("ephi_cholera_2022", PAGE))

4 hours of treatment) or treatment failure (the patient is still dehydrated after completing the initial 4 hours of rehydration therapy), regardless of the degree of dehydration; and

* Patients with coexisting conditions (including pregnancy) or comorbidities (such as SAM, HIV), regardless of the degree of dehydration.

*Antibiotics are given as soon as the patient is able to take oral medication (once vomiting has stopped).*

* Doxycycline — single dose (300 mg for adults; 2–4 mg/kg for children under 12 years of age) is the antibiotic of choice for all patients, including pregnant women.

* If resistance to doxycycline is documented, give azithromycin 1 g or ciprofloxacin 1 g orally as a single dose for adults.

* For children under 12 years of age, give azithromycin 20 mg/kg (max 1 g) or ciprofloxacin 20 mg/kg (max 1 g) orally as a single dose

|Age group|First- line|Alternative|
|-|-|-|
|Adults (including pregnant women)|Doxycycline 300 mg p.o. single dose|Azithromycin 1g p.o. sin

The table is now a table, one row per age group. We will start with the **simple pypdf text** (the obvious first
thing anyone would do), and in notebook 3 measure what the better parser buys us.

> **Try it.** Set `PAGE = 29` (the Ringer's lactate table) and run the last three cells again.

Now read all six guidelines with pypdf, one row per page:

In [5]:
rows = []
for pdf in sorted(rag.DOCS.glob("*.pdf")):
    for number, page in enumerate(PdfReader(pdf).pages, 1):
        text = re.sub(r"\s+", " ", page.extract_text() or "").strip()      # one line per page, single spaces
        if len(text.split()) >= 20:                                       # skip blank and image-only pages
            rows.append({"doc": pdf.stem, "page": number, "text": text})
pages = pd.DataFrame(rows)
print(f"{len(pages)} pages with text, {pages.text.str.split().str.len().sum():,} words")
pages.groupby("doc").size().rename("pages")

739 pages with text, 316,497 words


doc
ephi_cebs_2021                           50
ephi_cholera_2022                       121
ephi_climate_sensitive_diseases_2022    110
ephi_emergency_nutrition_2022            90
moh_acute_malnutrition_2019             153
moh_tb_2017                             215
Name: pages, dtype: int64

## Step 2. Chunk: from pages to passages

We will give the model only a few passages per question, not whole documents: a model can only read so much at
once (its *context window*), every extra word costs money and time, and too much text buries the answer. So we cut
the text into **chunks** (passages).

The simplest way: windows of 200 words. Consecutive windows **overlap** by 40 words, so a sentence cut at one
edge is whole in the next window. Each passage stays within one page, so we can always cite its page.

In [6]:
def chunk(text, size=200, overlap=40):
    """Cut a text into windows of `size` words; each starts `size - overlap` words after the previous one."""
    words = text.split()
    step = size - overlap
    return [" ".join(words[start:start + size]) for start in range(0, max(len(words) - overlap, 1), step)]


example = pages[(pages.doc == "ephi_cholera_2022") & (pages.page == PAGE)].text.iloc[0]
for i, c in enumerate(chunk(example)):
    print(f"--- passage {i} ({len(c.split())} words)\n{c[:300]} ... {c[-150:]}\n")

--- passage 0 (200 words)
31 Guideline for Cholera Surveillance and Outbreak Response Ethiopia 4 hours of treatment) or treatment failure (the patient is still dehydrated after completing the initial 4 hours of rehydration therapy), regardless of the degree of dehydration; and ■ Patients with coexisting conditions (including ... < 12 years old Doxycycline 2-4 mg/ kg p.o. single dose Azithromycin 20 mg/kg (max 1g) p.o. single dose, or ciprofloxacin 20 mg/kg (max 1g) p.o. single

--- passage 1 (186 words)
women) Doxycycline 300 mg p.o. single dose Azithromycin 1g p.o. single dose Children < 12 years old Doxycycline 2-4 mg/ kg p.o. single dose Azithromycin 20 mg/kg (max 1g) p.o. single dose, or ciprofloxacin 20 mg/kg (max 1g) p.o. single dose ■ The laboratory should monitor patterns of resistance of t ... luding ciprofloxacin. For the best effect with these classes of drugs, antibiotics should be administered 2 hours before zinc or 4–6 hours after zinc.



In [7]:
passages = pd.DataFrame([{"doc": p.doc, "page": p.page, "text": c}
                         for p in pages.itertuples() for c in chunk(p.text)])
print(f"{len(passages)} passages from {len(pages)} pages")
passages.text.str.split().str.len().describe().round()

2163 passages from 739 pages


count    2163.0
mean      173.0
std        47.0
min        25.0
25%       156.0
50%       200.0
75%       200.0
max       200.0
Name: text, dtype: float64

> **PAUSE (3 min).** What happens to a table that is longer than 200 words? To a sentence that runs from one page
> onto the next? What would change if passages were 50 words? 1,000 words?

Small passages are precise but can lose context (a dose without the age group it belongs to); large ones keep
context but dilute the search and cost more. There is no right size, only trade-offs you **measure** (notebook 3).

## Step 3. Index and retrieve: find the passages that match a question

The simplest search is by **keywords**. TF-IDF gives each word a weight: high if it appears often in a passage but
rarely across all passages ("doxycycline"), low if it is everywhere ("patient", "the"). A question and a passage
match if they share heavily weighted words. Building the index once takes a second.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english")   # words and word pairs
index = vectorizer.fit_transform(passages.text)                                             # one row per passage
print(f"index: {index.shape[0]} passages x {index.shape[1]:,} words and word pairs")


def search(question, k=5):
    """The k passages that share the most (rare) words with the question."""
    scores = (index @ vectorizer.transform([question]).T).toarray().ravel()
    best = scores.argsort()[::-1][:k]
    return passages.iloc[best].assign(score=scores[best].round(3))


search("A 7-year-old with cholera has stopped vomiting. Which antibiotic should I give, and at what dose?")

index: 2163 passages x 95,867 words and word pairs


,doc,page,text,score
139,ephi_cholera_2022,25,a rectal swab sample from the first 5 suspected cholera patients seen at the health facility. ■ Give an oral antibio...,0.134
168,ephi_cholera_2022,38,31 Guideline for Cholera Surveillance and Outbreak Response Ethiopia 4 hours of treatment) or treatment failure (the...,0.116
1858,moh_tb_2017,160,months. The monthly supervised dose is Rifampicin & Dapsone (R & DDS) and is taken at the start of treatment (day 1)...,0.083
1862,moh_tb_2017,161,months. The monthly supervised dose is Rifampicin & Dapsone (R & DDS) and is taken at the start of treatment (day 1)...,0.083
138,ephi_cholera_2022,25,protocols and train health personnel for implementation. Avail this guideline and any other simplified treatment pro...,0.079


> **Try it.** Search for a few questions of your own. When does keyword search fail? (Hint: what if the question
> uses different words from the guideline: "loose stools" instead of "diarrhoea", "baby" instead of "infant"?)

## Step 4. Generate: the LLM answers from the passages

The **prompt** is the instruction we send with every question. Ours is short and strict:

In [9]:
SYSTEM_PROMPT = rag.PROMPTS["v1"]                 # versioned: "v1". A new wording is a new version of the assistant
print(SYSTEM_PROMPT)

You answer questions from health workers in Ethiopia using ONLY the numbered guideline passages provided.
- Base every statement on the passages and cite them like [1] or [2][3].
- If the passages do not contain the answer, reply exactly: "The guidelines provided do not cover this."
- Be concise: at most 4 sentences. Quote numbers, doses and cut-offs exactly.
- Answer in the language of the question.


In [10]:
def ask(question, k=5, verbose=True):
    hits = search(question, k)
    numbered = "\n\n".join(f"[{i}] ({h.doc}, page {h.page}) {h.text}" for i, h in enumerate(hits.itertuples(), 1))
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Passages:\n\n{numbered}\n\nQuestion: {question}"}]
    reply = client.chat.completions.create(model=MODEL, temperature=0, messages=messages, **rag.thinking_switch(False))
    answer = reply.choices[0].message.content.strip()
    if verbose:
        show(answer)
        print("\nSources:")
        for i, h in enumerate(hits.itertuples(), 1):
            print(f"  [{i}] {h.doc}, page {h.page} (score {h.score})")
        print(f"\n{reply.usage.prompt_tokens} tokens in, {reply.usage.completion_tokens} out")
    return answer, hits


answer, hits = ask("A 7-year-old with cholera has stopped vomiting. Which antibiotic should I give, and at what dose?")

For a child under 12 years of age, the antibiotic of choice is a single dose of Doxycycline at 2–4 mg/kg [2].
If resistance to doxycycline is documented, you should give azithromycin 20 mg/kg (max 1 g) or ciprofloxacin
20 mg/kg (max 1 g) orally as a single dose [2]. Antibiotics should be given as soon as the patient is able to
take oral medication, once vomiting has stopped [2].

Sources:
  [1] ephi_cholera_2022, page 25 (score 0.134)
  [2] ephi_cholera_2022, page 38 (score 0.116)
  [3] moh_tb_2017, page 160 (score 0.083)
  [4] moh_tb_2017, page 161 (score 0.083)
  [5] ephi_cholera_2022, page 25 (score 0.079)

1579 tokens in, 107 out


Check the citation: open the passage it cites (`hits.text.iloc[0]` is passage [1]) and see whether it really
says that. Then ask something the guidelines do not cover:

In [11]:
_ = ask("What is the recommended dose of insulin for a newly diagnosed type 1 diabetic child?")

The guidelines provided do not cover this.

Sources:
  [1] moh_tb_2017, page 135 (score 0.086)
  [2] moh_tb_2017, page 134 (score 0.086)
  [3] moh_tb_2017, page 22 (score 0.086)
  [4] moh_tb_2017, page 23 (score 0.086)
  [5] moh_tb_2017, page 40 (score 0.081)

1396 tokens in, 9 out


It should refuse with the exact sentence from the prompt ("The guidelines provided do not cover this.").

**That is a complete RAG chatbot**: parse, chunk, index, retrieve, generate, in about 40 lines.

## Step 5. The chat page

The same five steps are packaged in `rag.py` (with settings you can change: which text, how to chunk, how to
search, how many passages, which model), and `chat_app.py` puts a web page on top. In a **terminal**, from this
folder, with the environment active and the key set:

```bash
python chat_app.py          # then open http://127.0.0.1:8000
```

For every question the page shows the answer, every passage the search returned (click a citation like **[2]** to
jump to it), and the exact prompt sent to the model. Leave the setup on **PDF text / fixed / keywords**: that is
what we just built.

> **Explore (15 min), in pairs.** Try these, then your own. For each, look at the passages, not only the answer.
> 1. What is the dose of doxycycline for an adult with cholera?
> 2. How much Ringer's lactate for a severely dehydrated 3-year-old, and over what time? (a table)
> 3. Which interventions are needed when acute malnutrition is 12% with no aggravating factors? (a table)
> 4. When is a child with a cough a presumptive TB case?
> 5. What should a community volunteer report, and to whom?
> 6. What is the treatment for malaria in pregnancy? (are the guidelines about this?)
> 7. Ask one question in Amharic.
>
> **Write down one answer you think is wrong or incomplete, and why.** Was it the search (wrong passages) or the
> model (right passages, wrong answer)? We need that distinction in notebook 2.